In [8]:
# Instala dependências necessárias no kernel do notebook
%pip install 'matplotlib>=3.7' -q

Note: you may need to restart the kernel to use updated packages.


In [10]:
# Verifica se os arquivos de dados necessários existem
import os
required = [
    "../data/olist_orders_dataset.csv",
    "../data/olist_order_items_dataset.csv",
    "../data/olist_order_payments_dataset.csv",
    "../data/olist_order_reviews_dataset.csv",
    "../data/olist_customers_dataset.csv",
    "../data/olist_products_dataset.csv",
    "../data/olist_sellers_dataset.csv",
    "../data/product_category_name_translation.csv",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print("Arquivos de dados ausentes:")
    for m in missing:
        print(" -", m)
    raise FileNotFoundError("Faltam arquivos de dados. Coloque os CSVs em `data/` ou atualize os caminhos.")
else:
    print("Todos os arquivos de dados necessários estão presentes.")

Arquivos de dados ausentes:
 - ../data/olist_orders_dataset.csv
 - ../data/olist_order_items_dataset.csv
 - ../data/olist_order_payments_dataset.csv
 - ../data/olist_order_reviews_dataset.csv
 - ../data/olist_customers_dataset.csv
 - ../data/olist_products_dataset.csv
 - ../data/olist_sellers_dataset.csv
 - ../data/product_category_name_translation.csv


FileNotFoundError: Faltam arquivos de dados. Coloque os CSVs em `data/` ou atualize os caminhos.

# Análise Exploratória — Brazilian E-Commerce Public Dataset by Olist

**Tech Challenge — Fase 1 — IA para Devs**
**Case:** Olist Intelligent Marketplace

Este notebook documenta, passo a passo, a análise exploratória usada como base para o
relatório executivo e o Mapa de Agentes de IA propostos para a Olist. Os dados são
reais, baixados diretamente do [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
(99.441 pedidos, 2016–2018).

Além das métricas, este notebook inclui gráficos para cada bloco de análise, tornando os
achados mais fáceis de visualizar e comunicar.


## 1. Setup e carregamento dos dados

Carregamos as tabelas de pedidos, itens, pagamentos, avaliações, clientes, produtos e vendedores, e configuramos o estilo dos gráficos.

In [9]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", None)

NAVY = "#1B2A4A"
TEAL = "#0F6E6A"
PINK = "#C2185B"
GRAY = "#8A94A6"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#D9DEE6",
    "axes.grid": True,
    "grid.color": "#EDEFF3",
    "grid.linewidth": 0.8,
    "font.size": 11,
})

orders = pd.read_csv("../data/olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date"
])
items = pd.read_csv("../data/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
customers = pd.read_csv("../data/olist_customers_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
cat_translation = pd.read_csv("../data/product_category_name_translation.csv")

orders.head()

Matplotlib is building the font cache; this may take a moment.


FileNotFoundError: [Errno 2] No such file or directory: '../data/olist_orders_dataset.csv'

## 2. Volumetria geral

Quantos pedidos, clientes, vendedores e produtos existem na base, e qual o período coberto.

In [ ]:
print("Pedidos:", orders["order_id"].nunique())
print("Clientes únicos:", customers["customer_unique_id"].nunique())
print("Vendedores:", sellers["seller_id"].nunique())
print("Produtos:", products["product_id"].nunique())
print("Período:", orders["order_purchase_timestamp"].min(), "a", orders["order_purchase_timestamp"].max())
print()
print("Distribuição de status do pedido:")
orders["order_status"].value_counts()

## 3. Prazo de entrega e atraso

Calculamos o tempo de entrega (dias entre compra e entrega) e o atraso (entrega real vs. estimada), apenas para pedidos com status `delivered`.

In [ ]:
delivered = orders[orders["order_status"] == "delivered"].copy()
delivered["delivery_days"] = (delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]).dt.days
delivered["delay_days"] = (delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]).dt.days
delivered["is_late"] = delivered["delay_days"] > 0

print(f"Prazo médio de entrega: {delivered['delivery_days'].mean():.1f} dias (mediana {delivered['delivery_days'].median():.0f})")
print(f"Pedidos atrasados: {100*delivered['is_late'].mean():.2f}%")
print(f"Atraso médio quando atrasado: {delivered.loc[delivered['is_late'], 'delay_days'].mean():.1f} dias")

**Gráfico 1** — Distribuição do prazo de entrega (em dias), com a mediana marcada.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.hist(delivered["delivery_days"].clip(upper=60), bins=40, color=TEAL, edgecolor="white")
ax.axvline(delivered["delivery_days"].median(), color=PINK, linestyle="--", linewidth=1.6,
           label=f"Mediana: {delivered['delivery_days'].median():.0f} dias")
ax.set_title("Distribuição do prazo de entrega", fontsize=13, fontweight="bold", color=NAVY)
ax.set_xlabel("Dias entre compra e entrega")
ax.set_ylabel("Nº de pedidos")
ax.legend(frameon=False)
plt.show()

## 4. Reviews: nota média e efeito do atraso

Este é um dos achados mais importantes: cruzamos pontualidade da entrega com a nota dada pelo cliente.

In [ ]:
print(f"Nota média geral: {reviews['review_score'].mean():.2f}")
print(f"Avaliações 1-2 estrelas: {100*reviews['review_score'].isin([1,2]).mean():.2f}%")
print(f"Avaliações com comentário escrito: {100*reviews['review_comment_message'].notna().mean():.2f}%")
print()

rev_orders = delivered.merge(reviews[["order_id", "review_score"]], on="order_id", how="inner")
score_by_lateness = rev_orders.groupby("is_late")["review_score"].mean().round(2)
print("Nota média — pedido no prazo:", score_by_lateness.get(False))
print("Nota média — pedido atrasado:", score_by_lateness.get(True))

**Gráfico 2** — Distribuição das notas de avaliação (1 a 5 estrelas).

In [ ]:
counts = reviews["review_score"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(counts.index.astype(str), counts.values, color=[PINK, PINK, GRAY, TEAL, TEAL])
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 800, f"{b.get_height():,}",
            ha="center", fontsize=9, color=NAVY)
ax.set_title("Distribuição das notas de avaliação (reviews)", fontsize=13, fontweight="bold", color=NAVY)
ax.set_xlabel("Nota (estrelas)")
ax.set_ylabel("Nº de avaliações")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.show()

**Gráfico 3** — Comparação direta da nota média entre pedidos no prazo e atrasados. Este é o achado mais visual do diagnóstico.

In [ ]:
means = rev_orders.groupby("is_late")["review_score"].mean()
labels = ["No prazo", "Atrasado"]
values = [means.get(False, 0), means.get(True, 0)]

fig, ax = plt.subplots(figsize=(5.5, 4.2))
bars = ax.bar(labels, values, color=[TEAL, PINK], width=0.55)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.08, f"{v:.2f}", ha="center", fontsize=12, fontweight="bold", color=NAVY)
ax.set_ylim(0, 5.4)
ax.set_title("Nota média: pedido no prazo vs. atrasado", fontsize=13, fontweight="bold", color=NAVY)
ax.set_ylabel("Nota média (1-5)")
plt.show()

## 5. Taxa de recompra

O achado mais impactante do diagnóstico. Usamos `customer_unique_id` (identifica a pessoa) em vez de `customer_id` (identifica o registro do pedido, que muda a cada compra).

In [ ]:
orders_cust = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
purchase_counts = orders_cust.groupby("customer_unique_id")["order_id"].nunique()

print(f"Clientes com apenas 1 compra: {100*(purchase_counts == 1).mean():.2f}%")
print(f"Clientes com recompra (2+): {100*(purchase_counts > 1).mean():.2f}%")
print(f"Média de pedidos por cliente: {purchase_counts.mean():.3f}")

**Gráfico 4** — Proporção de clientes que compraram uma única vez versus os que recompraram.

In [ ]:
single = (purchase_counts == 1).sum()
repeat = (purchase_counts > 1).sum()

fig, ax = plt.subplots(figsize=(5.5, 4.2))
wedges, texts, autotexts = ax.pie(
    [single, repeat], labels=["Compra única", "Recompra (2+)"], colors=[GRAY, TEAL],
    autopct="%1.1f%%", startangle=90, textprops={"fontsize": 11},
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
for t in autotexts:
    t.set_color("white")
    t.set_fontweight("bold")
ax.set_title("Clientes: compra única vs. recompra", fontsize=13, fontweight="bold", color=NAVY)
plt.show()

## 6. Pagamentos e catálogo de produtos

In [ ]:
print(f"Ticket médio: R$ {payments['payment_value'].mean():.2f}")
print(f"Parcelamento médio: {payments['payment_installments'].mean():.2f}x")
print()
print("Formas de pagamento:")
print(payments["payment_type"].value_counts())

**Gráfico 5** — Distribuição das formas de pagamento utilizadas.

In [ ]:
counts = payments["payment_type"].value_counts()
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.bar(counts.index, counts.values, color=TEAL)
ax.set_title("Distribuição das formas de pagamento", fontsize=13, fontweight="bold", color=NAVY)
ax.set_ylabel("Nº de transações")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.show()

In [ ]:
cat_map = dict(zip(cat_translation["product_category_name"], cat_translation["product_category_name_english"]))
top_cats = items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
top5 = top_cats["product_category_name"].value_counts().head(5)
top5.index = [cat_map.get(c, c) for c in top5.index]
print(f"Total de categorias: {products['product_category_name'].nunique()}")
print("\nTop 5 categorias por volume de itens vendidos:")
top5

**Gráfico 6** — Top 8 categorias de produto por volume de itens vendidos.

In [ ]:
top8 = top_cats["product_category_name"].value_counts().head(8)
top8.index = [cat_map.get(c, c) for c in top8.index]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.barh(top8.index[::-1], top8.values[::-1], color=NAVY)
ax.set_title("Top 8 categorias de produto por volume de itens vendidos", fontsize=13, fontweight="bold", color=NAVY)
ax.set_xlabel("Nº de itens vendidos")
plt.show()

## 7. Distribuição geográfica e atraso por estado

Cruzamos o estado do cliente com prazo médio de entrega e taxa de atraso — usado para priorizar o monitoramento logístico por região no Agente de Logística.

In [ ]:
top_states = customers["customer_state"].value_counts().head(5)
print("Top 5 estados por volume de pedidos:")
print(top_states)
print()

d_state = delivered.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
late_by_state = d_state.groupby("customer_state")["is_late"].mean().mul(100).round(2)
delivery_by_state = d_state.groupby("customer_state")["delivery_days"].mean().round(1)

resumo = pd.DataFrame({
    "pedidos": top_states,
    "atraso_%": [late_by_state.get(s) for s in top_states.index],
    "prazo_medio_dias": [delivery_by_state.get(s) for s in top_states.index],
})
resumo

**Gráfico 7** — Taxa de atraso por estado, para os 6 estados de maior volume de pedidos.

In [ ]:
top6_states = customers["customer_state"].value_counts().head(6).index.tolist()
late_by_state_6 = d_state.groupby("customer_state")["is_late"].mean().mul(100).reindex(top6_states)

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(top6_states, late_by_state_6.values, color=PINK)
for b, v in zip(bars, late_by_state_6.values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.2, f"{v:.1f}%", ha="center", fontsize=10, color=NAVY)
ax.set_title("Taxa de atraso por estado (top 6 em volume de pedidos)", fontsize=13, fontweight="bold", color=NAVY)
ax.set_ylabel("% de pedidos atrasados")
plt.show()

## 8. Síntese dos achados

| Dimensão | Achado |
|---|---|
| Prazo de entrega | ~12,1 dias em média; 6,8% dos pedidos atrasam |
| Nota vs. atraso | Cai de 4,28 (no prazo) para 2,26 (atrasado) |
| Reviews | Nota média 4,07; 15,1% são 1-2 estrelas; 41,8% têm comentário |
| Recompra | Apenas 3,1% dos clientes compram mais de uma vez |
| Geografia | RJ tem o maior atraso (12,1%) e maior prazo médio (14,8 dias) entre os 5 maiores estados |

Esses achados embasam diretamente o **Mapa de Agentes de IA** proposto no relatório executivo:
Agente de Atendimento, Agente de Reviews, Agente de Logística e Agente de Churn — cada um
endereçando uma das oportunidades identificadas aqui.

Os gráficos deste notebook também estão disponíveis como arquivos PNG individuais em
`results/figures/`, gerados por `src/generate_charts.py`.

Repositório e relatório completo: ver `README.md`.
